In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================

SILVER = "workspace.firmable_v2_silver"
OUTPUT_TABLE = f"{SILVER}.manager_candidates"

# ============================================================
# 2. READ SILVER EMPLOYEES
# ============================================================

employees = spark.table(
    f"{SILVER}.employees_clean"
).select(
    "company_id",
    "employee_id",
    "job_title_clean",
    "department_clean",
    "business_unit_clean",
    "team_id_clean",
    "team_name_clean",
    "inferred_seniority_level",
    "management_authority",
    "title_function",
    "org_structure"
)

# ============================================================
# 3. MAP DEPARTMENTS TO EXECUTIVE FUNCTIONS
# ============================================================

dept = F.coalesce(
    F.col("department_clean"),
    F.lit("")
)

employees = employees.withColumn(
    "department_function",
    F.when(
        dept.isin(
            "Engineering & Technology",
            "Data & Analytics",
            "Security & It",
            "Product Management"
        ),
        "Technology"
    )
    .when(
        dept == "Finance & Accounting",
        "Finance"
    )
    .when(
        dept.isin(
            "Operations",
            "Procurement & Supply Chain"
        ),
        "Operations"
    )
    .when(
        dept.isin(
            "Sales & Business Development",
            "Customer Success & Support",
            "Marketing"
        ),
        "Sales"
    )
    .when(
        dept == "Human Resources",
        "Human Resources"
    )
    .otherwise(F.lit(None).cast("string"))
)

# ============================================================
# 4. PREPARE EMPLOYEE-MANAGER PAIRS
# ============================================================

e = employees.alias("e")
m = employees.alias("m")

pair_condition = (
    (F.col("e.company_id") == F.col("m.company_id")) &
    (F.col("e.employee_id") != F.col("m.employee_id")) &
    F.col("e.inferred_seniority_level").isNotNull() &
    F.col("m.inferred_seniority_level").isNotNull() &
    (
        F.col("m.inferred_seniority_level") <
        F.col("e.inferred_seniority_level")
    )
)

pairs = e.join(
    m,
    pair_condition,
    "inner"
)

# ============================================================
# 5. DEFINE MATCHING EVIDENCE
# ============================================================

same_team = (
    F.col("e.team_id_clean").isNotNull() &
    (F.col("e.team_id_clean") == F.col("m.team_id_clean"))
)

same_department = (
    F.col("e.department_clean").isNotNull() &
    (F.col("e.department_clean") == F.col("m.department_clean"))
)

same_business_unit = (
    F.col("e.business_unit_clean").isNotNull() &
    (F.col("e.business_unit_clean") == F.col("m.business_unit_clean"))
)

same_executive_function = (
    F.col("e.department_function").isNotNull() &
    (
        F.col("e.department_function") ==
        F.col("m.title_function")
    )
)

manager_has_no_team = F.col("m.team_id_clean").isNull()

# ============================================================
# 6. CANDIDATE GENERATION HELPER
# ============================================================

def make_candidates(
    condition,
    relationship_type,
    priority
):
    return (
        pairs
        .filter(condition)
        .select(
            F.col("e.company_id").alias("company_id"),
            F.col("e.employee_id").alias("employee_id"),
            F.col("m.employee_id").alias("candidate_manager_id"),

            F.col("e.job_title_clean").alias("employee_title"),
            F.col("m.job_title_clean").alias("candidate_manager_title"),

            F.col("e.department_clean").alias("employee_department"),
            F.col("m.department_clean").alias("manager_department"),

            F.col("e.business_unit_clean").alias("employee_business_unit"),
            F.col("m.business_unit_clean").alias("manager_business_unit"),

            F.col("e.team_id_clean").alias("employee_team_id"),
            F.col("m.team_id_clean").alias("manager_team_id"),

            F.col("e.inferred_seniority_level").alias("employee_seniority"),
            F.col("m.inferred_seniority_level").alias("manager_seniority"),

            F.col("m.management_authority").alias("manager_authority"),

            F.col("e.org_structure").alias("org_structure"),

            F.lit(relationship_type).alias("relationship_type"),
            F.lit(priority).alias("candidate_priority")
        )
    )

# ============================================================
# 7. IC -> TEAM LEAD (PRIMARY)
# ============================================================

ic_team_lead = make_candidates(
    same_team &
    F.col("e.inferred_seniority_level").between(11, 13) &
    (F.col("m.inferred_seniority_level") == 10) &
    (
        F.col("m.management_authority") ==
        "Potential People Manager"
    ),
    "Team Lead to Individual Contributor",
    1
)

# ============================================================
# 8. IC -> TEAM MANAGER (FALLBACK ALTERNATIVE)
# Retain even when a Team Lead candidate exists; Notebook 06 can use it
# only if the higher-ranked candidate cannot be assigned.
# ============================================================

ic_team_manager = make_candidates(
    same_team &
    F.col("e.inferred_seniority_level").between(11, 13) &
    F.col("m.inferred_seniority_level").isin(8, 9) &
    (
        F.col("m.management_authority") ==
        "People Manager"
    ),
    "Team Manager to Individual Contributor",
    2
)

# ============================================================
# 9. IC -> DEPARTMENT HEAD (SKIP-LEVEL FALLBACK ALTERNATIVE)
# Retain alongside closer tiers; scoring and assignment govern its use.
# ============================================================

ic_department_head = make_candidates(
    same_department &
    manager_has_no_team &
    F.col("e.inferred_seniority_level").between(11, 13) &
    F.col("m.inferred_seniority_level").isin(5, 6, 7),
    "Department Head to Individual Contributor",
    3
)

# ============================================================
# 10. TEAM LEAD -> TEAM MANAGER (PRIMARY)
# ============================================================

lead_team_manager = make_candidates(
    same_team &
    (F.col("e.inferred_seniority_level") == 10) &
    F.col("m.inferred_seniority_level").isin(8, 9) &
    (
        F.col("m.management_authority") ==
        "People Manager"
    ),
    "Team Manager to Team Lead",
    1
)

# ============================================================
# 11. TEAM LEAD -> DEPARTMENT HEAD (FALLBACK)
# Also supports Flat companies where no team manager exists.
# ============================================================

# ============================================================
# 11. TEAM LEAD -> DEPARTMENT HEAD (FALLBACK)
# ============================================================

lead_department_head = make_candidates(
    same_department &
    manager_has_no_team &
    (F.col("e.inferred_seniority_level") == 10) &
    F.col("m.inferred_seniority_level").isin(5, 6, 7),
    "Department Head to Team Lead",
    2
).withColumn(
    "candidate_priority",
    F.when(
        F.col("org_structure") == "Flat",
        1
    ).otherwise(2)
)

# ============================================================
# 12. TEAM MANAGER -> DEPARTMENT HEAD
# ============================================================

team_manager_department_head = make_candidates(
    same_department &
    manager_has_no_team &
    F.col("e.inferred_seniority_level").isin(8, 9) &
    F.col("m.inferred_seniority_level").isin(5, 6, 7) &
    F.col("e.org_structure").isin(
        "Functional", "Divisional", "Hybrid"
    ),
    "Department Head to Team Manager",
    1
)

# ============================================================
# 13. DEPARTMENT HEAD -> FUNCTIONAL EXECUTIVE
# ============================================================

functional_department_head = make_candidates(
    F.col("e.org_structure").isin("Functional", "Hybrid") &
    F.col("e.inferred_seniority_level").isin(5, 6, 7) &
    (F.col("m.inferred_seniority_level") == 1) &
    (F.col("m.management_authority") == "Executive") &
    same_executive_function,
    "Functional Executive to Department Head",
    1
)

# ============================================================
# 14. DEPARTMENT HEAD -> DIVISION PRESIDENT
# ============================================================

divisional_department_head = make_candidates(
    (F.col("e.org_structure") == "Divisional") &
    F.col("e.inferred_seniority_level").isin(5, 6, 7) &
    (F.col("m.inferred_seniority_level") == 2) &
    (F.col("m.management_authority") == "Executive") &
    same_business_unit,
    "Division President to Department Head",
    1
)

# ============================================================
# 15. DEPARTMENT HEAD -> CEO (FALLBACK)
# Supports missing executive-function or division evidence.
# ============================================================

department_head_ceo_fallback = make_candidates(
    F.col("e.org_structure").isin(
        "Functional", "Hybrid", "Divisional"
    ) &
    F.col("e.inferred_seniority_level").isin(5, 6, 7) &
    (F.col("m.inferred_seniority_level") == 0) &
    (F.col("m.management_authority") == "Executive"),
    "CEO to Department Head - Fallback",
    2
)

# ============================================================
# 16. FLAT: DEPARTMENT HEAD -> CEO
# ============================================================

flat_department_head = make_candidates(
    (F.col("e.org_structure") == "Flat") &
    F.col("e.inferred_seniority_level").isin(5, 6, 7) &
    (F.col("m.inferred_seniority_level") == 0) &
    (F.col("m.management_authority") == "Executive"),
    "CEO to Department Head - Flat",
    1
)

# ============================================================
# 17. C-SUITE -> CEO
# ============================================================

executive_to_ceo = make_candidates(
    (
        F.col("e.inferred_seniority_level") == 1
    ) &
    (
        F.col("m.inferred_seniority_level") == 0
    ) &
    (
        F.col("m.management_authority") == "Executive"
    ),
    "CEO to C-Suite Executive",
    1
)

# ============================================================
# 18. DIVISION PRESIDENT -> CEO
# ============================================================

president_to_ceo = make_candidates(
    (
        F.col("e.inferred_seniority_level") == 2
    ) &
    (
        F.col("m.inferred_seniority_level") == 0
    ) &
    (
        F.col("m.management_authority") == "Executive"
    ),
    "CEO to Division President",
    1
)

# ============================================================
# 19. COMBINE ALL CANDIDATES
# IMPORTANT: keep all eligible tiers. Do not anti-join away fallback options.
# ============================================================

candidates = (
    ic_team_lead
    .unionByName(ic_team_manager)
    .unionByName(ic_department_head)
    .unionByName(lead_team_manager)
    .unionByName(lead_department_head)
    .unionByName(team_manager_department_head)
    .unionByName(functional_department_head)
    .unionByName(divisional_department_head)
    .unionByName(department_head_ceo_fallback)
    .unionByName(flat_department_head)
    .unionByName(executive_to_ceo)
    .unionByName(president_to_ceo)
)

# ============================================================
# 20. DEDUPLICATE AND KEEP BEST RULE FOR EACH PAIR
# ============================================================

pair_window = (
    Window
    .partitionBy(
        "company_id",
        "employee_id",
        "candidate_manager_id"
    )
    .orderBy("candidate_priority")
)

candidates = (
    candidates
    .withColumn(
        "_rule_rank",
        F.row_number().over(pair_window)
    )
    .filter(F.col("_rule_rank") == 1)
    .drop("_rule_rank")
)

# ============================================================
# 21. VALIDATE CANDIDATE INTEGRITY BEFORE PUBLICATION
# ============================================================

invalid_rows = candidates.filter(
    F.col("company_id").isNull() |
    F.col("employee_id").isNull() |
    F.col("candidate_manager_id").isNull() |
    (F.col("employee_id") == F.col("candidate_manager_id")) |
    F.col("employee_seniority").isNull() |
    F.col("manager_seniority").isNull() |
    (F.col("manager_seniority") >= F.col("employee_seniority"))
)
invalid_count = invalid_rows.count()

duplicate_pairs = (
    candidates
    .groupBy("company_id", "employee_id", "candidate_manager_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if invalid_count > 0 or duplicate_pairs > 0:
    if invalid_count > 0:
        invalid_rows.show(20, truncate=False)
    raise ValueError(
        f"Candidate generation stopped: invalid_rows={invalid_count}, "
        f"duplicate_pairs={duplicate_pairs}. Output table was not overwritten."
    )

print("Pre-publication candidate checks passed.")
print("Invalid rows:", invalid_count)
print("Duplicate employee-manager pairs:", duplicate_pairs)

# ============================================================
# 22. COUNT ALTERNATIVE CANDIDATES
# ============================================================

candidate_window = Window.partitionBy(
    "company_id",
    "employee_id"
)

candidates = candidates.withColumn(
    "candidate_count",
    F.count("*").over(candidate_window)
)

# ============================================================
# 23. PUBLISH DELTA TABLE
# ============================================================

(
    candidates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("GLOBAL MULTI-TIER MANAGER CANDIDATE GENERATION COMPLETED")

print("Total employees:", employees.count())
print("Total candidate relationships:", candidates.count())

print("\nRelationship distribution:")

candidates.groupBy(
    "relationship_type"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

print("\nCandidate priority distribution:")

candidates.groupBy(
    "candidate_priority"
).count().orderBy(
    "candidate_priority"
).show(truncate=False)

print("\nCandidate count distribution:")

candidates.groupBy(
    "candidate_count"
).count().orderBy(
    "candidate_count"
).show(truncate=False)

# Employees without any candidate
unresolved = (
    employees.select("company_id", "employee_id")
    .join(
        candidates.select(
            "company_id",
            "employee_id"
        ).distinct(),
        ["company_id", "employee_id"],
        "left_anti"
    )
)

print(
    "\nEmployees without candidates:",
    unresolved.count()
)

print("\nAll eligible fallback tiers are retained as alternatives.")
print("Output table:", OUTPUT_TABLE)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. READ DATA
# ============================================================

employees = spark.table(
    "workspace.firmable_v2_silver.employees_clean"
)

candidates = spark.table(
    "workspace.firmable_v2_silver.manager_candidates"
)

# ============================================================
# 2. IDENTIFY EMPLOYEES WITHOUT CANDIDATES
# ============================================================

candidate_employee_ids = (
    candidates
    .select("company_id", "employee_id")
    .distinct()
)

unresolved = (
    employees
    .join(
        candidate_employee_ids,
        ["company_id", "employee_id"],
        "left_anti"
    )
)

print("TOTAL UNRESOLVED EMPLOYEES:", unresolved.count())

# ============================================================
# 3. BREAKDOWN BY ORGANIZATIONAL STRUCTURE
# ============================================================

print("\nUNRESOLVED BY ORGANIZATIONAL STRUCTURE")

unresolved.groupBy(
    "org_structure"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

# ============================================================
# 4. BREAKDOWN BY SENIORITY AND AUTHORITY
# ============================================================

print("\nUNRESOLVED BY SENIORITY AND AUTHORITY")

unresolved.groupBy(
    "inferred_seniority_level",
    "management_authority"
).count().orderBy(
    "inferred_seniority_level"
).show(50, truncate=False)

# ============================================================
# 5. BREAKDOWN BY JOB TITLE
# ============================================================

print("\nTOP UNRESOLVED JOB TITLES")

unresolved.groupBy(
    "job_title_clean"
).count().orderBy(
    F.desc("count")
).show(30, truncate=False)

# ============================================================
# 6. CHECK DEPARTMENT AND BUSINESS UNIT AVAILABILITY
# ============================================================

print("\nUNRESOLVED DATA QUALITY")

unresolved.agg(
    F.sum(
        F.col("department_clean").isNull().cast("int")
    ).alias("missing_department"),

    F.sum(
        F.col("business_unit_clean").isNull().cast("int")
    ).alias("missing_business_unit"),

    F.sum(
        F.col("team_id_clean").isNull().cast("int")
    ).alias("missing_team"),

    F.sum(
        F.col("inferred_seniority_level").isNull().cast("int")
    ).alias("unknown_seniority")
).show(truncate=False)

# ============================================================
# 7. CANDIDATE COUNT DISTRIBUTION
# ============================================================

print("\nCANDIDATE COUNT DISTRIBUTION")

candidates.groupBy(
    "candidate_count"
).count().orderBy(
    "candidate_count"
).show(truncate=False)

print("\nDIAGNOSTICS COMPLETED")

In [0]:
from pyspark.sql import functions as F

employees = spark.table(
    "workspace.firmable_v2_silver.employees_clean"
)

print("DEPARTMENT DISTRIBUTION")

employees.groupBy(
    "department_clean"
).count().orderBy(
    F.desc("count")
).show(50, truncate=False)

print("EXECUTIVE TITLE DISTRIBUTION")

employees.filter(
    F.col("inferred_seniority_level").isin(0, 1, 2)
).groupBy(
    "job_title_clean",
    "inferred_seniority_level"
).count().orderBy(
    "inferred_seniority_level",
    "job_title_clean"
).show(50, truncate=False)

print("UNKNOWN TITLE DISTRIBUTION")

employees.filter(
    F.col("inferred_seniority_level").isNull()
).groupBy(
    "job_title_clean"
).count().orderBy(
    F.desc("count")
).show(50, truncate=False)

In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.firmable_v2_silver"

employees = spark.table(
    f"{SILVER}.employees_clean"
)

candidates = spark.table(
    f"{SILVER}.manager_candidates"
)

# ============================================================
# 1. TRUE CANDIDATE COUNT PER EMPLOYEE
# ============================================================

employee_candidate_counts = (
    candidates
    .groupBy("company_id", "employee_id")
    .agg(F.countDistinct("candidate_manager_id").alias("candidate_count"))
)

print("EMPLOYEE-LEVEL CANDIDATE DISTRIBUTION")

employee_candidate_counts.groupBy(
    "candidate_count"
).count().orderBy(
    "candidate_count"
).show(truncate=False)

# ============================================================
# 2. IDENTIFY UNRESOLVED EMPLOYEES
# ============================================================

unresolved = (
    employees
    .join(
        employee_candidate_counts.select(
            "company_id", "employee_id"
        ),
        ["company_id", "employee_id"],
        "left_anti"
    )
)

print("TOTAL UNRESOLVED:", unresolved.count())

print("\nUNRESOLVED BY STRUCTURE")

unresolved.groupBy(
    "org_structure"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

print("\nUNRESOLVED BY SENIORITY")

unresolved.groupBy(
    "inferred_seniority_level",
    "management_authority"
).count().orderBy(
    "inferred_seniority_level"
).show(truncate=False)

print("\nUNRESOLVED JOB TITLES")

unresolved.groupBy(
    "job_title_clean"
).count().orderBy(
    F.desc("count")
).show(30, truncate=False)

# ============================================================
# 3. CANDIDATE RELATIONSHIP INTEGRITY
# ============================================================

print("\nCANDIDATE INTEGRITY CHECKS")

candidates.agg(
    F.count("*").alias("total_candidate_rows"),

    F.countDistinct(
        F.concat_ws(
            "_",
            "company_id",
            "employee_id",
            "candidate_manager_id"
        )
    ).alias("unique_candidate_pairs"),

    F.sum(
        (F.col("employee_id") == F.col("candidate_manager_id")).cast("int")
    ).alias("self_manager_relationships"),

    F.sum(
        (
            F.col("manager_seniority") >=
            F.col("employee_seniority")
        ).cast("int")
    ).alias("invalid_seniority_relationships")
).show(truncate=False)

# ============================================================
# 4. PRIORITY DISTRIBUTION BY RELATIONSHIP
# ============================================================

print("\nRELATIONSHIP PRIORITY DISTRIBUTION")

candidates.groupBy(
    "relationship_type",
    "candidate_priority"
).count().orderBy(
    "relationship_type",
    "candidate_priority"
).show(50, truncate=False)

print("\nDIAGNOSTICS COMPLETED")

In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.firmable_v2_silver"

employees = spark.table(f"{SILVER}.employees_clean")
candidates = spark.table(f"{SILVER}.manager_candidates")

# Identify employees without candidates
candidate_employees = candidates.select(
    "company_id", "employee_id"
).distinct()

unresolved = employees.join(
    candidate_employees,
    ["company_id", "employee_id"],
    "left_anti"
)

# Company-level executive availability
company_executives = employees.groupBy(
    "company_id"
).agg(
    F.sum(
        (F.col("inferred_seniority_level") == 0).cast("int")
    ).alias("ceo_count"),
    F.sum(
        (F.col("inferred_seniority_level") == 1).cast("int")
    ).alias("csuite_count"),
    F.sum(
        (F.col("inferred_seniority_level") == 2).cast("int")
    ).alias("division_president_count")
)

unresolved = unresolved.join(
    company_executives,
    "company_id",
    "left"
)

# ============================================================
# 1. CLASSIFY UNRESOLVED EMPLOYEES
# ============================================================

unresolved = unresolved.withColumn(
    "unresolved_reason",
    F.when(
        F.col("inferred_seniority_level") == 0,
        "CEO - Expected Root"
    )
    .when(
        F.col("job_title_clean").isNull(),
        "Missing Job Title"
    )
    .when(
        F.col("ceo_count") == 0,
        "CEO Candidate Missing"
    )
    .when(
        F.col("inferred_seniority_level").isin(1, 2),
        "Executive Relationship Unavailable"
    )
    .when(
        F.col("department_clean").isNull(),
        "Missing Department"
    )
    .when(
        F.col("team_id_clean").isNull(),
        "Missing Team"
    )
    .otherwise("Candidate Rule Coverage Gap")
)

print("UNRESOLVED REASON DISTRIBUTION")

unresolved.groupBy(
    "unresolved_reason"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

# ============================================================
# 2. INVESTIGATE LEVEL 8 AND 9 MANAGERS
# ============================================================

print("\nLEVEL 8 AND 9 MANAGER DIAGNOSTICS")

unresolved.filter(
    F.col("inferred_seniority_level").isin(8, 9)
).groupBy(
    "org_structure",
    "department_clean",
    "management_authority",
    "unresolved_reason"
).count().orderBy(
    F.desc("count")
).show(50, truncate=False)

# ============================================================
# 3. CHECK COMPANIES WITH MISSING CEO CANDIDATES
# ============================================================

print("\nCOMPANIES WITHOUT A RECOGNIZED CEO")

company_executives.filter(
    F.col("ceo_count") == 0
).groupBy(
    "csuite_count",
    "division_president_count"
).count().show(truncate=False)

# ============================================================
# 4. FINAL ROOT AND UNRESOLVED COUNTS
# ============================================================

print("\nFINAL UNRESOLVED SUMMARY")

unresolved.groupBy(
    "unresolved_reason",
    "org_structure"
).count().orderBy(
    "unresolved_reason",
    "org_structure"
).show(50, truncate=False)

print("\nDIAGNOSTIC COMPLETED")

In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.firmable_v2_silver"

employees = spark.table(
    f"{SILVER}.employees_clean"
)

candidates = spark.table(
    f"{SILVER}.manager_candidates"
)

# ============================================================
# 1. RECREATE UNRESOLVED EMPLOYEE SET
# ============================================================

candidate_ids = candidates.select(
    "company_id", "employee_id"
).distinct()

unresolved = employees.join(
    candidate_ids,
    ["company_id", "employee_id"],
    "left_anti"
)

# Identify companies without a recognized CEO.
ceo_companies = (
    employees
    .filter(F.col("inferred_seniority_level") == 0)
    .select("company_id")
    .distinct()
)

unresolved = unresolved.join(
    ceo_companies.withColumn("has_ceo", F.lit(True)),
    "company_id",
    "left"
)

# Isolate candidate-rule gaps.
gaps = unresolved.filter(
    F.col("job_title_clean").isNotNull() &
    F.col("department_clean").isNotNull() &
    F.col("has_ceo").isNotNull() &
    (F.col("inferred_seniority_level") != 0) &
    (
        ~F.col("inferred_seniority_level").isin(1, 2)
    )
)

print("CANDIDATE RULE GAPS:", gaps.count())

# ============================================================
# 2. CHECK POSSIBLE MANAGERS
# ============================================================

e = gaps.alias("e")
m = employees.alias("m")

possible_managers = e.join(
    m,
    (
        (F.col("e.company_id") == F.col("m.company_id")) &
        (F.col("e.employee_id") != F.col("m.employee_id")) &
        F.col("e.inferred_seniority_level").isNotNull() &
        F.col("m.inferred_seniority_level").isNotNull() &
        (
            F.col("m.inferred_seniority_level") <
            F.col("e.inferred_seniority_level")
        ) &
        F.col("m.management_authority").isin(
            "Executive",
            "People Manager",
            "Potential People Manager"
        )
    ),
    "left"
)

# ============================================================
# 3. SUMMARIZE POTENTIAL MANAGER EVIDENCE
# ============================================================

diagnostic = possible_managers.groupBy(
    F.col("e.company_id").alias("company_id"),
    F.col("e.employee_id").alias("employee_id")
).agg(
    F.first(F.col("e.job_title_clean")).alias("employee_title"),
    F.first(F.col("e.department_clean")).alias("department"),
    F.first(F.col("e.inferred_seniority_level")).alias("seniority"),
    F.first(F.col("e.org_structure")).alias("org_structure"),
    F.first(F.col("e.team_id_clean")).alias("team_id"),

    F.count(F.col("m.employee_id")).alias(
        "all_senior_manager_candidates"
    ),

    F.sum(
        F.when(
            F.col("m.team_id_clean").isNotNull() &
            (F.col("e.team_id_clean") == F.col("m.team_id_clean")) &
            F.col("m.management_authority").isin(
                "People Manager",
                "Potential People Manager"
            ),
            1
        ).otherwise(0)
    ).alias("same_team_manager_candidates"),

    F.sum(
    F.when(
        (
            F.col("e.department_clean").isNotNull()
        ) &
        (
            F.col("e.department_clean") ==
            F.col("m.department_clean")
        ) &
        (
            F.col("m.management_authority") == "People Manager"
        ),
        1
    ).otherwise(0)
).alias("same_department_manager_candidates"),

    F.sum(
        F.when(
            F.col("e.department_clean").isNotNull() &
            (F.col("e.department_clean") == F.col("m.department_clean")) &
            F.col("m.team_id_clean").isNull() &
            F.col("m.inferred_seniority_level").isin(5, 6, 7),
            1
        ).otherwise(0)
    ).alias("department_head_candidates")
)

# ============================================================
# 4. CLASSIFY WHY CANDIDATES MAY BE MISSING
# ============================================================

diagnostic = diagnostic.withColumn(
    "diagnostic_result",
    F.when(
        F.col("same_team_manager_candidates") > 0,
        "Same-team candidate excluded by existing rules"
    )
    .when(
        F.col("department_head_candidates") > 0,
        "Department-head candidate excluded by existing rules"
    )
    .when(
        F.col("same_department_manager_candidates") > 0,
        "Same-department manager available"
    )
    .when(
        F.col("all_senior_manager_candidates") > 0,
        "Only cross-department or executive alternatives available"
    )
    .otherwise("No eligible senior manager found")
)

# ============================================================
# 5. REVIEW DIAGNOSTIC RESULTS
# ============================================================

print("\nGAP CLASSIFICATION")

diagnostic.groupBy(
    "diagnostic_result"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

print("\nGAPS BY SENIORITY")

diagnostic.groupBy(
    "seniority",
    "diagnostic_result"
).count().orderBy(
    "seniority",
    F.desc("count")
).show(50, truncate=False)

print("\nTOP UNRESOLVED TITLES")

diagnostic.groupBy(
    "employee_title",
    "diagnostic_result"
).count().orderBy(
    F.desc("count")
).show(30, truncate=False)

print("\nDIAGNOSTIC COMPLETED")

In [0]:
%sql
WITH employee_candidates AS (
    SELECT
        company_id,
        employee_id,
        COUNT(DISTINCT candidate_manager_id) AS candidate_count
    FROM workspace.firmable_v2_silver.manager_candidates
    GROUP BY company_id, employee_id
)
SELECT
    COUNT(*) AS employees_with_candidates,
    SUM(CASE WHEN candidate_count = 1 THEN 1 ELSE 0 END)
        AS employees_with_one_candidate,
    SUM(CASE WHEN candidate_count > 1 THEN 1 ELSE 0 END)
        AS employees_with_multiple_candidates,
    MAX(candidate_count) AS maximum_candidates_per_employee,
    ROUND(AVG(candidate_count), 2) AS average_candidates_per_employee
FROM employee_candidates;